# Circulation data
Join all the files in the reanalysis, reduce to scope (10S to 10N, 180W to 20E), output single ds with U, W

In [1]:
import xarray as xr
import numpy as np
import os
from dask_jobqueue import PBSCluster
from dask.distributed import Client

In [2]:
cluster = PBSCluster(
    cores=4, # The number of cores you want
    memory='64GB', # Amount of memory (resource_spec is the one)
    processes=1, # How many processes
    queue='casper', # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    local_directory='$TMPDIR', # Use your local directory
    account='P93300313', # Input your project ID here
    walltime='12:00:00', # Amount of wall time
)
cluster.scale(jobs=15)
# Setup your client
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: http://128.117.208.173:8787/status,
Dashboard: http://128.117.208.173:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.173:38377,Workers: 0
Dashboard: http://128.117.208.173:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [3]:
# dates in scope 
hist_decades = ['191001-191912', '192001-192912',
                '193001-193912', '194001-194912', '195001-195912', '196001-196912',
                '197001-197912', '198001-198912', '199001-199912', '200001-200912', 
                '201001-201412']

# too many different member names (macro and micro perturbations)
macro_name = np.char.mod('%d', np.arange(1001, 1311, 10)).tolist()

micro_name = np.char.zfill(np.char.mod('%d', np.arange(1, 21, 1)), 3).tolist()


path = "/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/OMEGA/"
title = "b.e21.BHIST"
type = '.nc'

all_W_names = []
for a in ["cmip6", "smbb"]:
    hist = f'{title}{a}'
    for i in macro_name:
        for j in micro_name:
            member = []
            for z in hist_decades:
                file = path + hist + '.f09_g17.LE2-'+ i + '.' +  j + '.cam.h0.OMEGA.' + z + type
                # print(file)
                if os.path.exists(file):
                    member.append(file)
                else:
                    continue
            if member != []:
                all_W_names.append(member)

In [4]:
def pre_slice(ds):
    # select var
    ds = ds['OMEGA']
    # sort lon
    ds.coords['lon'] = (ds.coords['lon'] + 180) % 360 - 180
    ds = ds.sortby(ds.lon)
    # sort lat
    ds.coords['lat'] = (ds.coords['lat'] + 90) % 180 - 90
    ds = ds.sortby(ds.lat)
    scoped_ds = ds.sel(lat=slice(-10, 10), lon=slice(-180, 20))
    return scoped_ds

In [5]:
Wds = xr.open_mfdataset(all_W_names, combine='nested',
                        engine='h5netcdf', parallel=True,
                        concat_dim=['member', 'time'],
                        preprocess=pre_slice)
Wds

<xarray.DataArray 'OMEGA' (member: 100, time: 1260, lev: 32, lat: 22, lon: 161)> Size: 57GB
dask.array<concatenate, shape=(100, 1260, 32, 22, 161), dtype=float32, chunksize=(1, 1, 16, 12, 144), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) object 10kB 1910-02-01 00:00:00 ... 2015-01-01 00:00:00
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
Dimensions without coordinates: member
Attributes:
    mdims:         1
    units:         Pa/s
    long_name:     Vertical velocity (pressure)
    cell_methods:  time: mean

In [6]:
# This is to fix a no size calendar error that came from using zarr
# changed output to zarr due to file locking problems when saving from various workers
# the combination of loading with h5netcdf and saving to zarr seems to be the best
Wds.coords['time'] = Wds.coords['time'].compute()
Wds = Wds.assign_coords({'member': Wds.member})
Wds = Wds.chunk(chunks={'member': 1, 'time': -1, 'lev': -1, 'lat': -1, 'lon': -1})
Wds.encoding = {}
Wds['time'].encoding['units'] = 'days since 1850-01-01 00:00:00'
Wds['time'].encoding['calendar'] = 'noleap'
Wds['time'].encoding['dtype'] = 'int64'
Wds

<xarray.DataArray 'OMEGA' (member: 100, time: 1260, lev: 32, lat: 22, lon: 161)> Size: 57GB
dask.array<rechunk-merge, shape=(100, 1260, 32, 22, 161), dtype=float32, chunksize=(1, 1260, 32, 22, 161), chunktype=numpy.ndarray>
Coordinates:
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
  * time     (time) object 10kB 1910-02-01 00:00:00 ... 2015-01-01 00:00:00
  * lev      (lev) float64 256B 3.643 7.595 14.36 24.61 ... 957.5 976.3 992.6
  * lat      (lat) float64 176B -9.895 -8.953 -8.01 -7.068 ... 8.01 8.953 9.895
  * lon      (lon) float64 1kB -180.0 -178.8 -177.5 -176.2 ... 17.5 18.75 20.0
Attributes:
    mdims:         1
    units:         Pa/s
    long_name:     Vertical velocity (pressure)
    cell_methods:  time: mean

In [7]:
# all_Us = []
# for m in all_U_names:
#     member = []
#     for f in m:
#         # print(f)
#         ds = xr.open_dataset(f, engine='h5netcdf', chunks={})['U']
#         member.append(ds)
#     member_data = xr.concat(member, dim='time')
#     all_Us.append(member_data)

In [8]:
# Uds = xr.concat(all_Us, dim='member')
# Uds = Uds.chunk(chunks={'member': 2, 'time': 1260, 'lev': 32, 'lat': 192, 'lon': 288})
# Uds

In [9]:
Wds.to_zarr('/glade/work/acruz/CESM/CESM_tropic_W', mode='w', consolidated=True)

/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/distributed/client.py:3374: UserWarning: Sending large graph of size 714.52 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [10]:
client.shutdown()

In [11]:
quit()